## Complete Titanic Cleaning Code

In [3]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("Titanic-Dataset.csv")

# View first rows
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


#### 1. Check Dataset Size and Missing Values

In [4]:
print("Dataset shape:", df.shape)

missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

missing_summary

Dataset shape: (891, 12)


,Missing Values,Missing Percentage
PassengerId,0,0.000000
Survived,0,0.000000
Pclass,0,0.000000
Name,0,0.000000
Sex,0,0.000000
Age,177,19.865320
SibSp,0,0.000000
Parch,0,0.000000
Ticket,0,0.000000
Fare,0,0.000000


#### 2. Check Duplicates

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df = df.drop_duplicates()

#### 3. Clean Age

In [7]:
df["Age"] = df.groupby(["Sex", "Pclass"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

#### 4. Clean Embarked

In [8]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

#### 5. Clean Fare

In [9]:
df["Fare"] = df.groupby("Pclass")["Fare"].transform(
    lambda x: x.fillna(x.median())
)

#### 6. Clean Cabin

In [10]:
df["HasCabin"] = df["Cabin"].notna().astype(int)

df["Deck"] = df["Cabin"].str[0]
df["Deck"] = df["Deck"].fillna("Unknown")

df = df.drop(columns=["Cabin"])

#### 7. Create New Useful Columns

In [11]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

df["IsAlone"] = np.where(df["FamilySize"] == 1, 1, 0)

df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"]
)

df["FareGroup"] = pd.qcut(
    df["Fare"],
    q=4,
    labels=["Low Fare", "Medium Fare", "High Fare", "Very High Fare"]
)   

#### 8. Drop Columns That Are Not Needed for Analysis

In [12]:
df_cleaned = df.drop(columns=["PassengerId", "Name", "Ticket"])

#### 9. Final Check

In [13]:
df_cleaned.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
HasCabin      0
Deck          0
FamilySize    0
IsAlone       0
AgeGroup      0
FareGroup     0
dtype: int64

In [14]:
df_cleaned.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,Deck,FamilySize,IsAlone,AgeGroup,FareGroup
0,0,3,male,22.0,1,0,7.2500,S,0,Unknown,2,0,Young Adult,Low Fare
1,1,1,female,38.0,1,0,71.2833,C,1,C,2,0,Adult,Very High Fare
2,1,3,female,26.0,0,0,7.9250,S,0,Unknown,1,1,Young Adult,Medium Fare
3,1,1,female,35.0,1,0,53.1000,S,1,C,2,0,Young Adult,Very High Fare
4,0,3,male,35.0,0,0,8.0500,S,0,Unknown,1,1,Young Adult,Medium Fare


#### 10. Save Cleaned Dataset

In [15]:
df_cleaned.to_csv("titanic_cleaned.csv", index=False)